In [13]:
%pip install citylearn


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from citylearn.data import DataSet

zones = [f'citylearn_challenge_2020_climate_zone_{i}' for i in range(1, 5)]

paths = {}
for name in zones:
    paths[name] = DataSet().get_dataset(name, directory='./all_ds')
    print(name, '->', paths[name])

citylearn_challenge_2020_climate_zone_1 -> ./all_ds/citylearn_challenge_2020_climate_zone_1/schema.json
citylearn_challenge_2020_climate_zone_2 -> ./all_ds/citylearn_challenge_2020_climate_zone_2/schema.json
citylearn_challenge_2020_climate_zone_3 -> ./all_ds/citylearn_challenge_2020_climate_zone_3/schema.json
citylearn_challenge_2020_climate_zone_4 -> ./all_ds/citylearn_challenge_2020_climate_zone_4/schema.json


In [15]:
import json, shutil, numpy as np, pandas as pd

def add_pricing(zone, base='./all_ds', out='./priced'):
    src = f'{base}/citylearn_challenge_2020_climate_zone_{zone}'
    dst = f'{out}/zone{zone}_priced'
    shutil.copytree(src, dst, dirs_exist_ok=True)

    df = pd.read_csv(f'{dst}/Building_1.csv')
    hour, month = df['hour'].values, df['month'].values

    summer = (month >= 6) & (month <= 9)
    winter = (month == 12) | (month <= 2)

    off  = np.where(summer, 0.22, np.where(winter, 0.15, 0.18))
    mid  = np.where(summer, 0.40, np.where(winter, 0.25, 0.32))
    peak = np.where(summer, 0.70, np.where(winter, 0.40, 0.50))

    price = np.where((hour >= 17) & (hour <= 21), peak,
             np.where((hour >= 8) & (hour <= 16), mid, off))

    pd.DataFrame({
        'electricity_pricing': price,
        'electricity_pricing_predicted_1': np.roll(price, -6),
        'electricity_pricing_predicted_2': np.roll(price, -12),
        'electricity_pricing_predicted_3': np.roll(price, -24),
    }).to_csv(f'{dst}/pricing.csv', index=False)

    s = json.load(open(f'{dst}/schema.json'))
    for k in s['observations']:
        if 'pric' in k:
            s['observations'][k]['active'] = (k == 'electricity_pricing')
    for b in s['buildings'].values():
        b['pricing'] = 'pricing.csv'
    json.dump(s, open(f'{dst}/schema.json', 'w'), indent=2)

    return f'{dst}/schema.json'

schemas = {z: add_pricing(z) for z in range(1, 5)}
print(schemas)

{1: './priced/zone1_priced/schema.json', 2: './priced/zone2_priced/schema.json', 3: './priced/zone3_priced/schema.json', 4: './priced/zone4_priced/schema.json'}
